# Step 4 - Model Comparison

Compares two YOLOv8 models trained on the same football dataset

| Model | 
|-------|
| YOLOv8n (nano) |
| YOLOv8s (small) |

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import yaml
import torch
import ultralytics
from ultralytics import YOLO

In [3]:
DATASET_DIR = r'D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset'

YAML_PATH = os.path.join(DATASET_DIR, 'data.yaml')

NANO_RUN_DIR = os.path.join(DATASET_DIR, 'runs', 'sports_player_detection')

SMALL_RUN_DIR = os.path.join(DATASET_DIR, 'runs', 'sports_player_detection_small')

COMPARE_DIR = os.path.join(DATASET_DIR, 'comparison')
os.makedirs(COMPARE_DIR, exist_ok=True)

EPOCHS     = 50
IMAGE_SIZE = 416
BATCH_SIZE = 4

In [4]:
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16

print(f'Base model : yolov8s.pt')
print(f'Epochs     : {EPOCHS}')
print(f'Image size : {IMAGE_SIZE}')
print(f'Batch size : {BATCH_SIZE}')

small_model = YOLO('yolov8s.pt')

small_results = small_model.train(
    data      = YAML_PATH,
    epochs    = EPOCHS,
    imgsz     = IMAGE_SIZE,
    batch     = BATCH_SIZE,
    device    = 'cpu',
    project   = os.path.join(DATASET_DIR, 'runs'),
    name      = 'sports_player_detection_small',
    exist_ok  = True,
    patience  = 15,
    save      = True,
    plots     = True,
    verbose   = True,
)

Base model : yolov8s.pt
Epochs     : 50
Image size : 640
Batch size : 16
New https://pypi.org/project/ultralytics/8.4.78 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.75  Python-3.10.0 torch-2.11.0+cpu CPU (13th Gen Intel Core i7-13620H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, ma

In [5]:
nano_weights  = os.path.join(NANO_RUN_DIR,  'weights', 'best.pt')
small_weights = os.path.join(SMALL_RUN_DIR, 'weights', 'best.pt')

print('Evaluating YOLOv8n on test set')
nano_model   = YOLO(nano_weights)
nano_metrics = nano_model.val(
    data    = YAML_PATH,
    split   = 'test',
    device  = 'cpu',
    verbose = False,
)


print('\nEvaluating YOLOv8s on test set')
small_model   = YOLO(small_weights)
small_metrics = small_model.val(
    data    = YAML_PATH,
    split   = 'test',
    device  = 'cpu',
    verbose = False,
)

Evaluating YOLOv8n on test set
Ultralytics 8.4.75  Python-3.10.0 torch-2.11.0+cpu CPU (13th Gen Intel Core i7-13620H)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 378.294.5 MB/s, size: 325.6 KB)
val: Scanning D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\labels\test.cache... 48 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 48/48  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.9it/s 1.6s1.0s
                   all         48        614      0.931       0.86      0.884      0.651
Speed: 0.1ms preprocess, 13.4ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\Code_PAS02\runs\detect\val-2

Evaluating YOLOv8s on test set
Ultralytics 8.4.75  Python-3.10.0 torch-2.11.0+cpu CPU (13th Gen Intel Core i7-13620H)
Model summary (fused): 73 layers, 11

In [6]:

comparison = pd.DataFrame([
    {
        'Model'          : 'YOLOv8n (nano)',
        'Image Size'     : IMAGE_SIZE,
        'Epochs'         : EPOCHS,
        'Precision'      : round(nano_metrics.box.mp,    4),
        'Recall'         : round(nano_metrics.box.mr,    4),
        'mAP@0.50'       : round(nano_metrics.box.map50, 4),
        'mAP@0.50:0.95'  : round(nano_metrics.box.map,   4),
        'Speed (ms/img)' : round(nano_metrics.speed.get('inference', 0), 2),
    },
    {
        'Model'          : 'YOLOv8s (small)',
        'Image Size'     : IMAGE_SIZE,
        'Epochs'         : EPOCHS,
        'Precision'      : round(small_metrics.box.mp,    4),
        'Recall'         : round(small_metrics.box.mr,    4),
        'mAP@0.50'       : round(small_metrics.box.map50, 4),
        'mAP@0.50:0.95'  : round(small_metrics.box.map,   4),
        'Speed (ms/img)' : round(small_metrics.speed.get('inference', 0), 2),
    },
])

print('MODEL COMPARISON TABLE')
print(comparison.to_string(index=False))

csv_out = os.path.join(COMPARE_DIR, 'comparison_results.csv')
comparison.to_csv(csv_out, index=False)

MODEL COMPARISON TABLE
          Model  Image Size  Epochs  Precision  Recall  mAP@0.50  mAP@0.50:0.95  Speed (ms/img)
 YOLOv8n (nano)         640      50     0.9309  0.8599    0.8842         0.6510           13.44
YOLOv8s (small)         640      50     0.9493  0.8730    0.9124         0.7238           75.11


In [8]:
metrics_to_plot = ['Precision', 'Recall', 'mAP@0.50', 'mAP@0.50:0.95']
x = np.arange(len(metrics_to_plot))
width = 0.35

nano_vals  = [comparison.loc[0, m] for m in metrics_to_plot]
small_vals = [comparison.loc[1, m] for m in metrics_to_plot]

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, nano_vals,  width, label='YOLOv8n (nano)',  color='#2196F3', edgecolor='white')
bars2 = ax.bar(x + width/2, small_vals, width, label='YOLOv8s (small)', color='#4CAF50', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_title('YOLOv8n vs YOLOv8s — Detection Performance on Football Dataset',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
out1 = os.path.join(COMPARE_DIR, '01_metric_comparison.png')
plt.savefig(out1, dpi=150)
plt.show()

<Figure size 1000x600 with 1 Axes>

In [9]:
nano_csv  = os.path.join(NANO_RUN_DIR,  'results.csv')
small_csv = os.path.join(SMALL_RUN_DIR, 'results.csv')

if os.path.exists(nano_csv) and os.path.exists(small_csv):
    df_nano  = pd.read_csv(nano_csv)
    df_small = pd.read_csv(small_csv)

    df_nano.columns  = df_nano.columns.str.strip()
    df_small.columns = df_small.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Training Curves: YOLOv8n vs YOLOv8s', fontsize=13, fontweight='bold')

    axes[0].plot(df_nano['epoch'],  df_nano['train/box_loss'],  label='YOLOv8n train', color="#C408D2", linewidth=2)
    axes[0].plot(df_nano['epoch'],  df_nano['val/box_loss'],    label='YOLOv8n val',   color='#C408D2', linewidth=2, linestyle='--')
    axes[0].plot(df_small['epoch'], df_small['train/box_loss'], label='YOLOv8s train', color="#265BC4", linewidth=2)
    axes[0].plot(df_small['epoch'], df_small['val/box_loss'],   label='YOLOv8s val',   color='#265BC4', linewidth=2, linestyle='--')
    axes[0].set_title('Box Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=8)
    axes[0].grid(linestyle='--', alpha=0.4)

    # mAP@0.5
    axes[1].plot(df_nano['epoch'],  df_nano['metrics/mAP50(B)'],  label='YOLOv8n', color='#C408D2', linewidth=2)
    axes[1].plot(df_small['epoch'], df_small['metrics/mAP50(B)'], label='YOLOv8s', color="#265BC4", linewidth=2)
    axes[1].set_title('mAP@0.50 over Epochs')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('mAP@0.50')
    axes[1].legend(fontsize=9)
    axes[1].grid(linestyle='--', alpha=0.4)

    # Precision
    axes[2].plot(df_nano['epoch'],  df_nano['metrics/precision(B)'],  label='YOLOv8n', color='#C408D2', linewidth=2)
    axes[2].plot(df_small['epoch'], df_small['metrics/precision(B)'], label='YOLOv8s', color="#265BC4", linewidth=2)
    axes[2].set_title('Precision over Epochs')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Precision')
    axes[2].legend(fontsize=9)
    axes[2].grid(linestyle='--', alpha=0.4)

    plt.tight_layout()
    out2 = os.path.join(COMPARE_DIR, '02_loss_curves.png')
    plt.savefig(out2, dpi=150)
    plt.show()

else:
    print('results.csv not found in one or both run directories.')
    print(f'  Looking for: {nano_csv}')
    print(f'  Looking for: {small_csv}')

<Figure size 1800x500 with 3 Axes>

In [11]:
nano_pr  = os.path.join(NANO_RUN_DIR,  'BoxPR_curve.png')
small_pr = os.path.join(SMALL_RUN_DIR, 'BoxPR_curve.png')

if os.path.exists(nano_pr) and os.path.exists(small_pr):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Precision-Recall Curves', fontsize=13, fontweight='bold')

    axes[0].imshow(mpimg.imread(nano_pr))
    axes[0].set_title('YOLOv8n (nano)', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(mpimg.imread(small_pr))
    axes[1].set_title('YOLOv8s (small)', fontsize=12, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    out3 = os.path.join(COMPARE_DIR, '03_pr_curves.png')
    plt.savefig(out3, dpi=150)
    plt.show()
else:
    print('PR curve PNGs not found — skipping this plot.')
    print(f'  Looking for: {nano_pr}')
    print(f'  Looking for: {small_pr}')

<Figure size 1600x600 with 2 Axes>

In [16]:

fig, ax = plt.subplots(figsize=(8, 6))

models      = ['YOLOv8n\n(nano)', 'YOLOv8s\n(small)']
map50_vals  = [comparison.loc[0, 'mAP@0.50'], comparison.loc[1, 'mAP@0.50']]
speed_vals  = [comparison.loc[0, 'Speed (ms/img)'], comparison.loc[1, 'Speed (ms/img)']]
colors      = ['#C408D2', '#265BC4']
sizes       = [300, 500]  

for i, (model, speed, acc, color, size) in enumerate(
    zip(models, speed_vals, map50_vals, colors, sizes)
):
    ax.scatter(speed, acc, s=size, color=color, alpha=0.85,
               edgecolors='black', linewidth=1.5, zorder=3)
    ax.annotate(
        model,
        (speed, acc),
        textcoords='offset points',
        xytext=(15, 5),
        fontsize=11,
        fontweight='bold',
        color=color,
    )

ax.set_xlabel('Inference Time (ms/image) — lower is faster', fontsize=11)
ax.set_ylabel('mAP@0.50 — higher is better', fontsize=11)
ax.set_title('Speed vs Accuracy Tradeoff\n(bubble size = model size)',
             fontsize=12, fontweight='bold')
ax.grid(linestyle='--', alpha=0.4)

plt.tight_layout()
out4 = os.path.join(COMPARE_DIR, '04_speed_vs_accuracy.png')
plt.savefig(out4, dpi=150)
plt.show()

<Figure size 800x600 with 1 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 1 Axes>

In [13]:

print('FINAL COMPARISON')
print(comparison.to_string(index=False))
print()

# Auto-determine winner per metric
for metric in ['Precision', 'Recall', 'mAP@0.50', 'mAP@0.50:0.95']:
    winner_idx = comparison[metric].idxmax()
    winner     = comparison.loc[winner_idx, 'Model']
    diff       = abs(comparison.loc[0, metric] - comparison.loc[1, metric])
    print(f'  {metric:20s}: {winner} wins  (diff: {diff:.4f})')

speed_winner_idx = comparison['Speed (ms/img)'].idxmin()
speed_winner     = comparison.loc[speed_winner_idx, 'Model']
print(f'  {"Inference Speed":20s}: {speed_winner} wins (faster)')

print()
print('Report files saved to:', COMPARE_DIR)
print('  01_metric_comparison.png')
print('  02_loss_curves.png')
print('  03_pr_curves.png')
print('  04_speed_vs_accuracy.png')
print('  comparison_results.csv')

FINAL COMPARISON
          Model  Image Size  Epochs  Precision  Recall  mAP@0.50  mAP@0.50:0.95  Speed (ms/img)
 YOLOv8n (nano)         640      50     0.9309  0.8599    0.8842         0.6510           13.44
YOLOv8s (small)         640      50     0.9493  0.8730    0.9124         0.7238           75.11

  Precision           : YOLOv8s (small) wins  (diff: 0.0184)
  Recall              : YOLOv8s (small) wins  (diff: 0.0131)
  mAP@0.50            : YOLOv8s (small) wins  (diff: 0.0282)
  mAP@0.50:0.95       : YOLOv8s (small) wins  (diff: 0.0728)
  Inference Speed     : YOLOv8n (nano) wins (faster)

Report files saved to: D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\comparison
  01_metric_comparison.png
  02_loss_curves.png
  03_pr_curves.png
  04_speed_vs_accuracy.png
  comparison_results.csv
